In [ ]:
import re
from firebase_util import FirestoreBaseUtil
from pydantic import BaseModel, Field, ConfigDict, model_validator
from datetime import datetime
from typing import Any, Annotated
from copy import deepcopy

# PROJECT_ID = "wilde-rosen-483d1"
PROJECT_ID = "wilde-rosen-npr-dfafc"

# Set up validator
util = FirestoreBaseUtil(PROJECT_ID)

In [ ]:
def url_to_path(url: str) -> str | None:
    regex = r'(https:\/\/firebasestorage\.googleapis\.com\/v0\/b\/)(.*)(\.firebasestorage\.app\/o\/)([^\?]*)(\??)(.*)'
    match = re.match(regex, url)
    if match is not None and len(match.groups()) >= 4:
        return match.group(4)
    return None


class DamageDetail(BaseModel):
    imagePath: str = Field(validation_alias="path")
    description: str
    # imageUrl: Annotated[str, Field(deprecated='Deprecated since commit 61623484')] = ""

    model_config = ConfigDict(validate_by_alias=True, validate_by_name=True)

    @model_validator(mode='before')
    @classmethod
    def check_card_number_not_present(cls, data: Any) -> Any:
        if isinstance(data, dict):
            if "imagePath" not in data or "path" not in data and "imageUrl" in data:
                path = url_to_path(data["imageUrl"])
                if path is not None:
                    d = deepcopy(data)
                    d["imagePath"] = path
                    return d
        return data


class DamageEntry(BaseModel):
    createdAt: datetime
    description: str
    details: list[DamageDetail]
    imagePath: str = Field(validation_alias="path")
    order: float | int
    side: str
    updatedAt: datetime
    x: float | int
    y: float | int
    # imageUrl: Annotated[str, Field(deprecated='Deprecated since commit 61623484')] = ""
    # schematicUrl: Annotated[str, Field(deprecated='Deprecated since commit 61623484')] = ""

    model_config = ConfigDict(validate_by_alias=True, validate_by_name=True)

    @model_validator(mode='before')
    @classmethod
    def check_card_number_not_present(cls, data: Any) -> Any:
        if isinstance(data, dict):
            if "imagePath" not in data or "path" not in data and "imageUrl" in data:
                path = url_to_path(data["imageUrl"])
                if path is not None:
                    d = deepcopy(data)
                    d["imagePath"] = path
                    return d
        return data

class Car(BaseModel):
    title: str

In [ ]:
damages_raw = util.db.collection_group("damages").stream()
for doc in damages_raw:
    damage = DamageEntry.model_validate(doc.to_dict())
    doc.reference.set(damage.model_dump())